In [3]:
import wfdb
import pandas as pd

In [4]:
# Load the Excel file (adjust path and sheet name as needed)
df_time = pd.read_excel('ReportHome75h.xlsx')

def get_start_timestamp(df, filename):
    # Filter the row with the filename
    row = df[df.iloc[:, 0] == filename]
    if row.empty:
        raise ValueError(f"Filename {filename} not found in data")
    
    # Extract date and time columns (adjust column names if different)
    timestamp_str = row.iloc[:, [3,4]]
    date_str = timestamp_str['0 day'].iloc[0]       # first row date
    time_str = timestamp_str['Unnamed: 4'].iloc[0]  # first row time

    # Convert to datetime
    timestamp = pd.to_datetime(f"{date_str} {time_str}")
    return timestamp
    

In [5]:
# file names from CO-001 to CO-044 and FL-001 to FL-036
file_names = [f"CO-{i:03d}" for i in range(1, 45)] + [f"FL-{i:03d}" for i in range(1, 37)]

for file_name in file_names:
    try:
        path = f"physionet.org/files/ltmm/1.0.0/{file_name.replace('-', '')}"
        output_path = f"minute_level/{file_name.replace('-', '')}.csv"
    
        # Read the record (without specifying extension)
        record = wfdb.rdrecord(path)
        
        # Convert to pandas DataFrame
        df = pd.DataFrame(record.p_signal, columns=record.sig_name)
        
        # Keep only the three acceleration columns
        acc_df = df[['v-acceleration', 'ml-acceleration', 'ap-acceleration']]

        try:
            start_ts = get_start_timestamp(df_time, file_name)

        except Exception as e:
            print(f"Could not find start timestamp for {file_name}: {e}")
            start_ts = pd.Timestamp('1970-01-01 00:00:00')

        acc_df = acc_df[['v-acceleration', 'ml-acceleration', 'ap-acceleration']].copy()
    
        sampling_rate = 100
        acc_df['time_s'] = acc_df.index / sampling_rate
        
        start_time = pd.Timestamp(start_ts, tz='UTC')
        acc_df['timestamp'] = start_time + pd.to_timedelta(acc_df.index / sampling_rate, unit='s')
        
        acc_df = acc_df.drop(columns=['time_s'])
        acc_df = acc_df.set_index('timestamp')
        
        acc_df = acc_df.rename(columns={
            'ml-acceleration': 'x',
            'ap-acceleration': 'y',
            'v-acceleration': 'z'
        })
        
        acc_df = acc_df[['x', 'y', 'z']]
        
        acc_df = acc_df.resample('1min').mean()
        
        # Convert index (DatetimeIndex) to ISO 8601 strings with milliseconds and Z
        acc_df['timestamp'] = acc_df.index.to_series().dt.strftime('%Y-%m-%d %H:%M:%S')#.str[:-3] + 'Z'
        
        acc_df = acc_df[['timestamp','x', 'y', 'z']]
    
        acc_df.to_csv(output_path, index=False)

    except Exception as e:
        print(f"{file_name} error: {e}")

Could not find start timestamp for CO-018: Filename CO-018 not found in data
Could not find start timestamp for CO-025: Filename CO-025 not found in data
CO-026 error: [Errno 2] No such file or directory: '/Users/jacquesleooscar/Documents/Education/ETHZ/Curriculum/Semester04/04MasterThesis/LTMM_Analysis/physionet.org/files/ltmm/1.0.0/CO026.hea'


/var/folders/d3/9hnj267n62j2wqlp3s_r48nw0000gp/T/ipykernel_41916/1625956932.py:16: UserWarning: Parsing dates in %d/%m/%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  timestamp = pd.to_datetime(f"{date_str} {time_str}")


CO-033 error: [Errno 2] No such file or directory: '/Users/jacquesleooscar/Documents/Education/ETHZ/Curriculum/Semester04/04MasterThesis/LTMM_Analysis/physionet.org/files/ltmm/1.0.0/CO033.hea'
CO-034 error: [Errno 2] No such file or directory: '/Users/jacquesleooscar/Documents/Education/ETHZ/Curriculum/Semester04/04MasterThesis/LTMM_Analysis/physionet.org/files/ltmm/1.0.0/CO034.hea'
Could not find start timestamp for CO-036: Unknown datetime string format, unable to parse: 2012-06-07 00:00:00 ?, at position 0
Could not find start timestamp for CO-038: Unknown datetime string format, unable to parse: nan nan, at position 0
CO-043 error: [Errno 2] No such file or directory: '/Users/jacquesleooscar/Documents/Education/ETHZ/Curriculum/Semester04/04MasterThesis/LTMM_Analysis/physionet.org/files/ltmm/1.0.0/CO043.hea'
Could not find start timestamp for CO-044: Filename CO-044 not found in data
Could not find start timestamp for FL-001: Filename FL-001 not found in data
FL-002 error: [Errno 2]